In [11]:

# LIBRERÍAS Y RUTAS

import pandas as pd
import numpy as np
from pathlib import Path
import pyarrow.parquet as pq

# Ruta actual
PROJECT_DIR = Path.cwd()

# Si el notebook se ejecuta desde Transform_data, subimos a la raíz del proyecto
if PROJECT_DIR.name == "Transform_data":
    PROJECT_DIR = PROJECT_DIR.parent

RAW_FILE = PROJECT_DIR / "data" / "raw" / "datacenter_enriched_core_single.parquet"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Ruta del proyecto:")
print(PROJECT_DIR)

print("\nArchivo base:")
print(RAW_FILE)

print("\nCarpeta de salida:")
print(PROCESSED_DIR)

print("\n¿Existe el archivo base?")
print(RAW_FILE.exists())

Ruta del proyecto:
d:\GB TI\Master\Visu_DATOS\Proyecto\project_hpc_visdatos\VisDatos_HPC

Archivo base:
d:\GB TI\Master\Visu_DATOS\Proyecto\project_hpc_visdatos\VisDatos_HPC\data\raw\datacenter_enriched_core_single.parquet

Carpeta de salida:
d:\GB TI\Master\Visu_DATOS\Proyecto\project_hpc_visdatos\VisDatos_HPC\data\processed

¿Existe el archivo base?
True


In [12]:

# REVISIÓN DE METADATOS DEL DATASET

pf = pq.ParquetFile(RAW_FILE)

print("Número de filas:")
print(pf.metadata.num_rows)

print("\nNúmero de columnas:")
print(len(pf.schema.names))

print("\nColumnas disponibles:")
for i, col in enumerate(pf.schema.names, start=1):
    print(f"{i}. {col}")

Número de filas:
11930727

Número de columnas:
53

Columnas disponibles:
1. prom_id
2. timestamp
3. timestamp_seconds
4. timestamp_seconds_delta
5. node
6. rack_inferred
7. node_position_inferred
8. gpu_node
9. gpu_model
10. gpu_count
11. cpu_tdp_total
12. gpu_tdp_total
13. node_load1
14. node_load5
15. node_load15
16. node_load1_per_core
17. node_power_usage
18. node_rapl_package_power_sum
19. node_rapl_package_joules_total_sum_delta
20. cpu_package_energy_kwh
21. node_memory_Active_bytes
22. node_memory_MemFree_bytes
23. ram_active_gb
24. ram_free_gb
25. node_hwmon_temp_celsius_mean
26. node_hwmon_temp_celsius_max
27. node_thermal_zone_temp_mean
28. node_thermal_zone_temp_max
29. nvidia_gpu_memory_used_bytes_sum
30. gpu_memory_used_gb
31. nvidia_gpu_temperature_celsius_mean
32. nvidia_gpu_temperature_celsius_max
33. nvidia_gpu_power_usage_milliwatts_mean
34. gpu_power_watts_mean
35. gpu_power_watts_max
36. nvidia_gpu_duty_cycle_mean
37. nvidia_gpu_duty_cycle_max
38. node_disk_read_by

In [13]:

# DATASET PARA SERIE TEMPORAL

cols_hourly = [
    "timestamp",
    "rack_inferred",
    "node",
    "gpu_node",
    "state",
    "node_power_usage",
    "node_energy_kwh_est",
    "node_hwmon_temp_celsius_mean",
    "node_hwmon_temp_celsius_max",
    "co2_kg_est"
]

df_hourly = pd.read_parquet(RAW_FILE, columns=cols_hourly)

# Conversión de fechas
df_hourly["timestamp"] = pd.to_datetime(df_hourly["timestamp"])

# Conversión recomendada: W a kW
df_hourly["node_power_kw"] = df_hourly["node_power_usage"] / 1000

# Agregación por hora
df_hourly["hour"] = df_hourly["timestamp"].dt.floor("h")

hourly_node_metrics = (
    df_hourly.groupby(["hour", "rack_inferred", "node", "gpu_node", "state"], dropna=False)
    .agg(
        power_mean_kw=("node_power_kw", "mean"),
        energy_kwh=("node_energy_kwh_est", "sum"),
        temp_mean_c=("node_hwmon_temp_celsius_mean", "mean"),
        temp_max_c=("node_hwmon_temp_celsius_max", "max"),
        co2_kg=("co2_kg_est", "sum"),
        records=("timestamp", "size")
    )
    .reset_index()
)

output_path = PROCESSED_DIR / "hourly_node_metrics.parquet"
hourly_node_metrics.to_parquet(output_path, index=False)

print("Archivo creado:")
print(output_path)

print("\nDimensiones:")
print(hourly_node_metrics.shape)

display(hourly_node_metrics.head())

Archivo creado:
d:\GB TI\Master\Visu_DATOS\Proyecto\project_hpc_visdatos\VisDatos_HPC\data\processed\hourly_node_metrics.parquet

Dimensiones:
(83629, 11)


,hour,rack_inferred,node,gpu_node,state,power_mean_kw,energy_kwh,temp_mean_c,temp_max_c,co2_kg,records
0,2022-06-30 16:00:00,r25,r25n30,0,FAILED,0.086000,0.005733,37.091250,48.0,0.002293,8
1,2022-06-30 17:00:00,r25,r25n30,0,CANCELLED,0.107091,0.039267,41.287273,57.0,0.015707,44
2,2022-06-30 17:00:00,r25,r25n30,0,COMPLETED,0.111492,0.058533,42.645714,58.0,0.023413,63
3,2022-06-30 18:00:00,r25,r25n30,0,COMPLETED,0.104211,0.099000,38.494561,56.0,0.039600,114
4,2022-06-30 18:00:00,r26,r26n15,0,COMPLETED,0.105763,0.052000,34.052034,51.0,0.020800,59


In [14]:

# DATASET PARA HEATMAP RACK / HORA


df_heat = hourly_node_metrics.copy()

df_heat["hour_of_day"] = df_heat["hour"].dt.hour

heatmap_rack_hour = (
    df_heat.groupby(["rack_inferred", "hour_of_day"], dropna=False)
    .agg(
        energy_kwh=("energy_kwh", "sum"),
        power_mean_kw=("power_mean_kw", "mean"),
        temp_mean_c=("temp_mean_c", "mean"),
        temp_max_c=("temp_max_c", "max"),
        co2_kg=("co2_kg", "sum")
    )
    .reset_index()
)

output_path = PROCESSED_DIR / "heatmap_rack_hour.parquet"
heatmap_rack_hour.to_parquet(output_path, index=False)

print("Archivo creado:")
print(output_path)

print("\nDimensiones:")
print(heatmap_rack_hour.shape)

display(heatmap_rack_hour.head())

Archivo creado:
d:\GB TI\Master\Visu_DATOS\Proyecto\project_hpc_visdatos\VisDatos_HPC\data\processed\heatmap_rack_hour.parquet

Dimensiones:
(336, 7)


,rack_inferred,hour_of_day,energy_kwh,power_mean_kw,temp_mean_c,temp_max_c,co2_kg
0,r10,0,57.399700,0.128566,43.646183,69.0,22.959880
1,r10,1,57.019067,0.132254,44.424104,70.0,22.807627
2,r10,2,56.528667,0.132556,44.507814,68.0,22.611467
3,r10,3,56.128400,0.129653,43.962691,69.0,22.451360
4,r10,4,61.494633,0.119498,41.916114,67.0,24.597853


In [15]:
# DATASET PARA RANKING POR RACK O NODO


ranking_rack_node = (
    hourly_node_metrics.groupby(["rack_inferred", "node", "gpu_node"], dropna=False)
    .agg(
        energy_kwh=("energy_kwh", "sum"),
        power_mean_kw=("power_mean_kw", "mean"),
        temp_mean_c=("temp_mean_c", "mean"),
        temp_max_c=("temp_max_c", "max"),
        co2_kg=("co2_kg", "sum"),
        records=("records", "sum")
    )
    .reset_index()
)

output_path = PROCESSED_DIR / "ranking_rack_node.parquet"
ranking_rack_node.to_parquet(output_path, index=False)

print("Archivo creado:")
print(output_path)

print("\nDimensiones:")
print(ranking_rack_node.shape)

display(ranking_rack_node.sort_values("energy_kwh", ascending=False).head(10))

Archivo creado:
d:\GB TI\Master\Visu_DATOS\Proyecto\project_hpc_visdatos\VisDatos_HPC\data\processed\ranking_rack_node.parquet

Dimensiones:
(62, 9)


,rack_inferred,node,gpu_node,energy_kwh,power_mean_kw,temp_mean_c,temp_max_c,co2_kg,records
49,r28,r28n1,1,3549.8768,1.005856,30.023696,46.0,1419.95072,416973
58,r34,r34n1,1,2948.5048,1.019059,29.538851,43.0,1179.40192,319698
50,r29,r29n3,1,2175.1756,0.974435,29.154763,44.0,870.07024,280190
59,r34,r34n2,1,1926.6740,1.006704,30.043562,44.0,770.66960,218601
51,r29,r29n4,1,1902.3062,1.148819,29.952602,43.0,760.92248,189331
60,r36,r36n2,1,1842.6722,0.886564,29.087459,43.0,737.06888,266002
61,r36,r36n3,1,1763.1448,1.001081,28.886997,41.0,705.25792,209864
56,r33,r33n3,1,1711.3720,0.831095,26.951398,32.0,684.54880,230805
53,r32,r32n4,1,1634.5542,0.770115,26.874086,33.0,653.82168,236812
52,r32,r32n3,1,1073.9220,0.721411,26.696422,32.0,429.56880,166342


In [16]:

# DATASET PARA CONSUMO POR ESTADO DE JOB


cols_state = [
    "slurm_id",
    "state",
    "gpu_node",
    "nodetypes",
    "node_energy_kwh_est",
    "node_power_usage",
    "job_duration_hours",
    "numcores",
    "numnodes"
]

df_state = pd.read_parquet(RAW_FILE, columns=cols_state)

# Conversión recomendada: W a kW
df_state["node_power_kw"] = df_state["node_power_usage"] / 1000

# Parte 1: métricas energéticas desde todos los registros
energy_state = (
    df_state.groupby(["state", "gpu_node", "nodetypes"], dropna=False)
    .agg(
        energy_kwh=("node_energy_kwh_est", "sum"),
        power_mean_kw=("node_power_kw", "mean"),
        records=("slurm_id", "size")
    )
    .reset_index()
)

# Parte 2: métricas de jobs evitando duplicar duración
job_unique = df_state.drop_duplicates(
    subset=["slurm_id", "state", "gpu_node", "nodetypes"]
)

job_state = (
    job_unique.groupby(["state", "gpu_node", "nodetypes"], dropna=False)
    .agg(
        jobs_count=("slurm_id", "nunique"),
        job_duration_mean_h=("job_duration_hours", "mean"),
        job_duration_max_h=("job_duration_hours", "max"),
        numcores_mean=("numcores", "mean"),
        numnodes_mean=("numnodes", "mean")
    )
    .reset_index()
)

# Unión de ambas partes
state_summary = energy_state.merge(
    job_state,
    on=["state", "gpu_node", "nodetypes"],
    how="left"
)

output_path = PROCESSED_DIR / "state_summary.parquet"
state_summary.to_parquet(output_path, index=False)

print("Archivo creado:")
print(output_path)

print("\nDimensiones:")
print(state_summary.shape)

display(state_summary.sort_values("energy_kwh", ascending=False).head(10))

Archivo creado:
d:\GB TI\Master\Visu_DATOS\Proyecto\project_hpc_visdatos\VisDatos_HPC\data\processed\state_summary.parquet

Dimensiones:
(130, 11)


,state,gpu_node,nodetypes,energy_kwh,power_mean_kw,records,jobs_count,job_duration_mean_h,job_duration_max_h,numcores_mean,numnodes_mean
56,COMPLETED,1,gpu_titanrtx_shared(1),5965.460600,1.006582,711145,1264,4.923341,100.890556,5.255538,1.0
128,TIMEOUT,1,gpu_titanrtx_shared(1),4809.262000,0.925066,623854,155,34.426586,120.008056,6.580645,1.0
26,CANCELLED,1,gpu_titanrtx_shared(1),3777.851200,1.134000,399772,445,7.628694,118.513056,6.970787,1.0
50,COMPLETED,1,gpu_shared(1),3270.337600,0.861302,455613,1908,2.049151,48.405278,3.854822,1.0
28,COMPLETED,0,normal(1),2985.379633,0.133054,2691959,16718,1.383646,120.835278,16.000000,1.0
41,COMPLETED,0,shared(1),1792.014567,0.129480,1660770,1774,8.248059,118.703333,6.042277,1.0
123,TIMEOUT,1,gpu_shared(1),1369.711800,0.855820,192051,76,22.078231,80.002222,4.697368,1.0
105,TIMEOUT,0,normal(1),1339.313200,0.128839,1247401,418,26.329740,120.007778,16.000000,1.0
116,TIMEOUT,0,shared(1),593.416400,0.117497,606041,441,13.125997,120.008056,1.718821,1.0
79,FAILED,1,gpu_titanrtx_shared(1),568.770200,1.096490,62244,367,1.433725,86.244722,5.795640,1.0


In [17]:

# DATASET DE DETALLE POR JOB Y NODO 

import pandas as pd
import numpy as np
import pyarrow.parquet as pq
from pathlib import Path
import shutil
import gc

cols_detail = [
    "slurm_id",
    "node",
    "rack_inferred",
    "gpu_node",
    "gpu_model",
    "gpu_count",
    "state",
    "nodetypes",
    "start_date",
    "end_date",
    "job_duration_hours",
    "numcores",
    "numnodes",
    "node_power_usage",
    "node_energy_kwh_est",
    "node_hwmon_temp_celsius_mean",
    "node_hwmon_temp_celsius_max",
    "co2_kg_est",
    "node_load1_per_core",
    "ram_active_gb",
    "gpu_memory_used_gb",
    "nvidia_gpu_duty_cycle_mean"
]

group_keys = [
    "slurm_id",
    "node",
    "rack_inferred",
    "gpu_node",
    "gpu_model",
    "gpu_count",
    "state",
    "nodetypes"
]

# Tamaño de bloque. Si vuelve a fallar por memoria, baja a 50_000.
BATCH_SIZE = 100_000

# Carpeta temporal para guardar agregados parciales
TEMP_DIR = PROCESSED_DIR / "_tmp_detail_parts"

if TEMP_DIR.exists():
    shutil.rmtree(TEMP_DIR)

TEMP_DIR.mkdir(parents=True, exist_ok=True)

pf = pq.ParquetFile(RAW_FILE)

print("Iniciando procesamiento por bloques...")
print(f"Filas totales aproximadas: {pf.metadata.num_rows:,}")
print(f"Tamaño de bloque: {BATCH_SIZE:,}")

part_number = 0

for batch in pf.iter_batches(batch_size=BATCH_SIZE, columns=cols_detail):
    chunk = batch.to_pandas()

    # Conversión de fechas
    chunk["start_date"] = pd.to_datetime(chunk["start_date"])
    chunk["end_date"] = pd.to_datetime(chunk["end_date"])

    # Conversión recomendada: W a kW
    chunk["node_power_kw"] = chunk["node_power_usage"] / 1000

    # Agregado parcial por job y nodo
    part = (
        chunk.groupby(group_keys, dropna=False)
        .agg(
            start_date=("start_date", "min"),
            end_date=("end_date", "max"),
            job_duration_hours=("job_duration_hours", "max"),
            numcores=("numcores", "max"),
            numnodes=("numnodes", "max"),

            power_sum_kw=("node_power_kw", "sum"),
            energy_kwh=("node_energy_kwh_est", "sum"),
            temp_mean_sum_c=("node_hwmon_temp_celsius_mean", "sum"),
            temp_max_c=("node_hwmon_temp_celsius_max", "max"),
            co2_kg=("co2_kg_est", "sum"),

            load1_per_core_sum=("node_load1_per_core", "sum"),
            ram_active_gb_sum=("ram_active_gb", "sum"),
            gpu_memory_used_gb_sum=("gpu_memory_used_gb", "sum"),
            gpu_duty_cycle_sum=("nvidia_gpu_duty_cycle_mean", "sum"),

            records=("slurm_id", "size")
        )
        .reset_index()
    )

    part_path = TEMP_DIR / f"detail_part_{part_number:04d}.parquet"
    part.to_parquet(part_path, index=False)

    print(f"Bloque {part_number + 1} procesado: {len(chunk):,} filas -> {len(part):,} filas agregadas")

    part_number += 1

    del chunk, part, batch
    gc.collect()

print("\nAgregados parciales creados.")
print(f"Total de bloques procesados: {part_number}")

Iniciando procesamiento por bloques...
Filas totales aproximadas: 11,930,727
Tamaño de bloque: 100,000
Bloque 1 procesado: 100,000 filas -> 270 filas agregadas
Bloque 2 procesado: 100,000 filas -> 221 filas agregadas
Bloque 3 procesado: 100,000 filas -> 309 filas agregadas
Bloque 4 procesado: 100,000 filas -> 230 filas agregadas
Bloque 5 procesado: 100,000 filas -> 188 filas agregadas
Bloque 6 procesado: 100,000 filas -> 309 filas agregadas
Bloque 7 procesado: 100,000 filas -> 515 filas agregadas
Bloque 8 procesado: 100,000 filas -> 379 filas agregadas
Bloque 9 procesado: 100,000 filas -> 316 filas agregadas
Bloque 10 procesado: 100,000 filas -> 359 filas agregadas
Bloque 11 procesado: 100,000 filas -> 144 filas agregadas
Bloque 12 procesado: 100,000 filas -> 448 filas agregadas
Bloque 13 procesado: 100,000 filas -> 386 filas agregadas
Bloque 14 procesado: 100,000 filas -> 198 filas agregadas
Bloque 15 procesado: 100,000 filas -> 152 filas agregadas
Bloque 16 procesado: 100,000 filas -

In [18]:

# UNIR AGREGADOS PARCIALES Y CREAR DATASET FINAL


partial_files = sorted(TEMP_DIR.glob("detail_part_*.parquet"))

print(f"Archivos parciales encontrados: {len(partial_files)}")

parts = []

for file in partial_files:
    parts.append(pd.read_parquet(file))

partial_detail = pd.concat(parts, ignore_index=True)

print("Dimensiones del agregado parcial unido:")
print(partial_detail.shape)

job_node_detail = (
    partial_detail.groupby(group_keys, dropna=False)
    .agg(
        start_date=("start_date", "min"),
        end_date=("end_date", "max"),
        job_duration_hours=("job_duration_hours", "max"),
        numcores=("numcores", "max"),
        numnodes=("numnodes", "max"),

        power_sum_kw=("power_sum_kw", "sum"),
        energy_kwh=("energy_kwh", "sum"),
        temp_mean_sum_c=("temp_mean_sum_c", "sum"),
        temp_max_c=("temp_max_c", "max"),
        co2_kg=("co2_kg", "sum"),

        load1_per_core_sum=("load1_per_core_sum", "sum"),
        ram_active_gb_sum=("ram_active_gb_sum", "sum"),
        gpu_memory_used_gb_sum=("gpu_memory_used_gb_sum", "sum"),
        gpu_duty_cycle_sum=("gpu_duty_cycle_sum", "sum"),

        records=("records", "sum")
    )
    .reset_index()
)

# Calcular promedios correctamente usando records
job_node_detail["power_mean_kw"] = job_node_detail["power_sum_kw"] / job_node_detail["records"]
job_node_detail["temp_mean_c"] = job_node_detail["temp_mean_sum_c"] / job_node_detail["records"]
job_node_detail["load1_per_core_mean"] = job_node_detail["load1_per_core_sum"] / job_node_detail["records"]
job_node_detail["ram_active_gb_mean"] = job_node_detail["ram_active_gb_sum"] / job_node_detail["records"]
job_node_detail["gpu_memory_used_gb_mean"] = job_node_detail["gpu_memory_used_gb_sum"] / job_node_detail["records"]
job_node_detail["gpu_duty_cycle_mean"] = job_node_detail["gpu_duty_cycle_sum"] / job_node_detail["records"]

# Eliminar columnas auxiliares
cols_to_drop = [
    "power_sum_kw",
    "temp_mean_sum_c",
    "load1_per_core_sum",
    "ram_active_gb_sum",
    "gpu_memory_used_gb_sum",
    "gpu_duty_cycle_sum"
]

job_node_detail = job_node_detail.drop(columns=cols_to_drop)

# Reordenar columnas
ordered_cols = [
    "slurm_id",
    "node",
    "rack_inferred",
    "gpu_node",
    "gpu_model",
    "gpu_count",
    "state",
    "nodetypes",
    "start_date",
    "end_date",
    "job_duration_hours",
    "numcores",
    "numnodes",
    "power_mean_kw",
    "energy_kwh",
    "temp_mean_c",
    "temp_max_c",
    "co2_kg",
    "load1_per_core_mean",
    "ram_active_gb_mean",
    "gpu_memory_used_gb_mean",
    "gpu_duty_cycle_mean",
    "records"
]

job_node_detail = job_node_detail[ordered_cols]

output_path = PROCESSED_DIR / "job_node_detail.parquet"
job_node_detail.to_parquet(output_path, index=False)

print("Archivo creado:")
print(output_path)

print("\nDimensiones finales:")
print(job_node_detail.shape)

display(job_node_detail.head())

# Limpiar temporales
del parts, partial_detail
gc.collect()

Archivos parciales encontrados: 120
Dimensiones del agregado parcial unido:
(33603, 23)
Archivo creado:
d:\GB TI\Master\Visu_DATOS\Proyecto\project_hpc_visdatos\VisDatos_HPC\data\processed\job_node_detail.parquet

Dimensiones finales:
(33344, 23)


,slurm_id,node,rack_inferred,gpu_node,gpu_model,gpu_count,state,nodetypes,start_date,end_date,...,power_mean_kw,energy_kwh,temp_mean_c,temp_max_c,co2_kg,load1_per_core_mean,ram_active_gb_mean,gpu_memory_used_gb_mean,gpu_duty_cycle_mean,records
0,1521049,r25n30,r25,0,n/a,0,FAILED,normal(1),2022-06-30 16:55:10,2022-06-30 16:59:18,...,0.086000,0.005733,37.091250,48.0,0.002293,0.608203,2.262334,0.0,0.0,8
1,1522340,r25n30,r25,0,n/a,0,CANCELLED,normal(1),2022-06-30 17:00:03,2022-06-30 17:09:31,...,0.101263,0.016033,44.018421,57.0,0.006413,0.405132,2.804049,0.0,0.0,19
2,1522345,r25n30,r25,0,n/a,0,CANCELLED,normal(1),2022-06-30 17:10:17,2022-06-30 17:17:55,...,0.119200,0.014900,39.963333,51.0,0.005960,0.227625,1.984719,0.0,0.0,15
3,1522349,r25n30,r25,0,n/a,0,CANCELLED,normal(1),2022-06-30 17:18:41,2022-06-30 17:23:57,...,0.100000,0.008333,38.084000,42.0,0.003333,0.202188,1.905713,0.0,0.0,10
4,1522354,r25n30,r25,0,n/a,0,COMPLETED,normal(1),2022-06-30 17:24:43,2022-06-30 17:36:49,...,0.098833,0.019767,43.437500,58.0,0.007907,0.391016,2.599730,0.0,0.0,24


0

In [19]:

# MUESTRA PARA SCATTERPLOT POTENCIA - TEMPERATURA


MAX_POINTS = 80000

if len(job_node_detail) > MAX_POINTS:
    scatter_sample = job_node_detail.sample(MAX_POINTS, random_state=42)
else:
    scatter_sample = job_node_detail.copy()

output_path = PROCESSED_DIR / "scatter_sample.parquet"
scatter_sample.to_parquet(output_path, index=False)

print("Archivo creado:")
print(output_path)

print("\nDimensiones:")
print(scatter_sample.shape)

display(scatter_sample.head())

Archivo creado:
d:\GB TI\Master\Visu_DATOS\Proyecto\project_hpc_visdatos\VisDatos_HPC\data\processed\scatter_sample.parquet

Dimensiones:
(33344, 23)


,slurm_id,node,rack_inferred,gpu_node,gpu_model,gpu_count,state,nodetypes,start_date,end_date,...,power_mean_kw,energy_kwh,temp_mean_c,temp_max_c,co2_kg,load1_per_core_mean,ram_active_gb_mean,gpu_memory_used_gb_mean,gpu_duty_cycle_mean,records
0,1521049,r25n30,r25,0,n/a,0,FAILED,normal(1),2022-06-30 16:55:10,2022-06-30 16:59:18,...,0.086000,0.005733,37.091250,48.0,0.002293,0.608203,2.262334,0.0,0.0,8
1,1522340,r25n30,r25,0,n/a,0,CANCELLED,normal(1),2022-06-30 17:00:03,2022-06-30 17:09:31,...,0.101263,0.016033,44.018421,57.0,0.006413,0.405132,2.804049,0.0,0.0,19
2,1522345,r25n30,r25,0,n/a,0,CANCELLED,normal(1),2022-06-30 17:10:17,2022-06-30 17:17:55,...,0.119200,0.014900,39.963333,51.0,0.005960,0.227625,1.984719,0.0,0.0,15
3,1522349,r25n30,r25,0,n/a,0,CANCELLED,normal(1),2022-06-30 17:18:41,2022-06-30 17:23:57,...,0.100000,0.008333,38.084000,42.0,0.003333,0.202188,1.905713,0.0,0.0,10
4,1522354,r25n30,r25,0,n/a,0,COMPLETED,normal(1),2022-06-30 17:24:43,2022-06-30 17:36:49,...,0.098833,0.019767,43.437500,58.0,0.007907,0.391016,2.599730,0.0,0.0,24


In [20]:

# VERIFICACIÓN FINAL DE ARCHIVOS PROCESADOS

processed_files = list(PROCESSED_DIR.glob("*.parquet"))

print("Archivos generados:")
for file in processed_files:
    size_mb = file.stat().st_size / (1024 ** 2)
    print(f"{file.name}: {size_mb:.2f} MB")

print("\nVista rápida de archivos:")
for file in processed_files:
    temp_df = pd.read_parquet(file)
    print(f"{file.name}: {temp_df.shape[0]} filas x {temp_df.shape[1]} columnas")

Archivos generados:
heatmap_rack_hour.parquet: 0.02 MB
hourly_node_metrics.parquet: 1.67 MB
job_node_detail.parquet: 2.08 MB
ranking_rack_node.parquet: 0.01 MB
scatter_sample.parquet: 2.08 MB
state_summary.parquet: 0.01 MB

Vista rápida de archivos:
heatmap_rack_hour.parquet: 336 filas x 7 columnas
hourly_node_metrics.parquet: 83629 filas x 11 columnas
job_node_detail.parquet: 33344 filas x 23 columnas
ranking_rack_node.parquet: 62 filas x 9 columnas
scatter_sample.parquet: 33344 filas x 23 columnas
state_summary.parquet: 130 filas x 11 columnas
